Start from the dataset kickstarter-14-04, take out the unnecessary columsn that I don't know why are there:

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import nltk
from nltk.corpus import stopwords
import string 
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import re 
from collections import Counter
import ast
import glob

In [2]:
campaigns = pd.read_csv('raw_kickstarter.csv', index_col=0)
campaigns

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD
...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD


### as a very first step we add the categories to each project

In [3]:

folder_path = "Kickstarter_2026-03-12T03_20_26_556Z"

files = glob.glob(f"{folder_path}/*.csv")

print("Number of files found:", len(files))

dfs = []

for file in files:
    print("Loading:", file)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)



url_to_category = {}
for _, row in df.iterrows():
    parsed = ast.literal_eval(row['category'])
    parent_name = parsed.get('parent_name') or parsed.get('name')
    url_parsed = ast.literal_eval(row['urls'])
    project_url = url_parsed['web']['project']
    url_to_category[project_url] = parent_name


campaigns['category'] = campaigns['url'].map(url_to_category)

print(f"Matched: {campaigns['category'].notna().sum()} / {len(campaigns)}")
print(campaigns['category'].value_counts())

Number of files found: 85
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter001.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter002.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter003.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter004.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter005.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter006.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter007.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter008.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter009.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter010.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter011.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter012.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter013.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter014.csv
Lo

take out the unnecessary columns, add the reached one and classify as success or failure (maybe it's already in the initial dataset but honestly I don't remember)

In [4]:
campaigns['reached'] = (campaigns['pledged'] / campaigns['goal']) * 100

median_reached = campaigns['reached'].median()
campaigns['status'] = campaigns['reached'].apply(lambda x: 1 if x >= median_reached else 0)

Preprocessing: usual preprocessing stuff like lowercasing, taking out links, only keepin alpha numeric characters, tokenizing the words, and taking out the english stopwords, also took out the words that are less than 2 characters

In [5]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text_p = "".join([char for char in text if char.isalnum() or char.isspace()])
    
    words = word_tokenize(text_p) 
    
    
    stop_words = stopwords.words('english')
    words = [w for w in words if w not in stop_words and len(w) > 2]
    filtered_words = [word for word in words if word not in stop_words] 
    
    return filtered_words  

campaigns['description_processed'] = campaigns['description'].apply(preprocess)

How do we decide which stopwords to include beside the 'normal' stopwords? We also want to include words that are very generic, not informative at all and coudl potentially appear in any kind of campaign, regardless of the topics in it (some examples could be the words 'kickstarter', campaign, etc etc)
Initially I thought TF-IDF would make sense but it doesn't actually, because TF-IDF both depends on how frequent a word is in each document and how frequent are documents that contain that words in the whole set of documents. (also you cannot average the TF-IDF of a word and pick the ones with lowest value because a rare word, which we wanna keep, could end up having a low TF-IDF value if it appears in few documents but many time in those few documents)
So potentially we can just use document frequency, meaning if the word appears in > alpha% of documents, we count it as a stopword?

Also quick sidenote: Other methods like BytePair encoding, WordPiece or SentencePiece are not really useful here because they look at subwords, and for example if we get 'kickstarter' and split it into kick and starter, then we might consider to keep kick because it makes sense with sport but we might not have any document where the word is present for real

This below just basically gives a counter of the document frequency of each token in our whole corpus of all the descriptions 

We apply lemmatization (I think probably better than stemming since it returns the lemma version of the word, instead of just cutting the ending of the word). We apply it now, before taking out the domain specific stopwords, because otherwise we could end up having to account for different versions of the same stopword (example: project vs projects)

In [6]:
lemmatizer = WordNetLemmatizer()

campaigns['lemmatized'] = campaigns['description_processed'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words])


In [7]:
docs = campaigns['lemmatized']
N = len(docs)

df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    'word': list(df_counter.keys()),
    'doc_freq': list(df_counter.values())
})

df_table['doc_freq_ratio'] = df_table['doc_freq'] / N
df_table = df_table.sort_values('doc_freq_ratio', ascending=False)



Now the only method and the one that made most sense to me to take out too common words is basically just defining a threshold and see which words are too rare (like if it appears only 2-3 times in the whole dataset) or too much (in this case too much can be like appearing in more than 40-60% of the descriptions), you can try different values of the two initial thresholds (the one I saw looks the best would be 0.0005 for the low and 0.55 for the high)

In [8]:
min_ratio = 0.0003
max_ratio = 0.50

vocab = set(
    df_table[
        (df_table['doc_freq_ratio'] > min_ratio) &
        (df_table['doc_freq_ratio'] < max_ratio)
    ]['word']
)

campaigns['description_processed'] = campaigns['lemmatized'].apply(
    lambda doc: [w for w in doc if w in vocab]
)

print(df_table[df_table['doc_freq_ratio'] >= max_ratio])
df_table[df_table['doc_freq_ratio'] <= min_ratio]
campaigns = campaigns.drop(columns=['lemmatized'])

             word  doc_freq  doc_freq_ratio
372           one      5194        0.706282
199          make      5190        0.705738
242          time      5127        0.697172
283          help      4925        0.669704
166       project      4743        0.644955
1019          new      4693        0.638156
128          like      4692        0.638020
341          also      4677        0.635980
825          year      4663        0.634077
33          first      4513        0.613680
402           get      4459        0.606337
605          need      4438        0.603481
273         world      4395        0.597634
633          work      4373        0.594642
234          life      4237        0.576149
0             way      4180        0.568398
420       support      4048        0.550449
349   kickstarter      4005        0.544602
339          goal      3952        0.537395
667          well      3860        0.524884
980          want      3813        0.518493
276        people      3717     

### Now we work on the data specific category by category 

In [9]:
df = campaigns

#### 1. Identification of words appearing in more than 40% of documents

In [10]:
if "description_filtered" not in df.columns:
    df["description_filtered"] = None


df_film = df[df['category'] == 'Film & Video'].copy()
docs = df_film["description_processed"]

N = len(docs)
df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    "word": list(df_counter.keys()),
    "doc_freq": list(df_counter.values())
})

df_table["doc_freq_ratio"] = df_table["doc_freq"] / N
high_freq_words = df_table[df_table["doc_freq_ratio"] > 0.5]
high_freq_words = high_freq_words.sort_values("doc_freq_ratio", ascending = False)
high_freq_words

,word,doc_freq,doc_freq_ratio
45,film,1689,0.839881
204,production,1333,0.662854
265,love,1062,0.528095
164,take,1045,0.519642
129,team,1024,0.509199
378,director,1024,0.509199
394,see,1014,0.504227
700,many,1012,0.503232


In [78]:
extra_stopwords_film = set(high_freq_words["word"].tolist())

film_mask = df["category"] == "Film & Video"

df.loc[film_mask, "description_filtered"] = df.loc[film_mask, "description_processed"].apply(
    lambda text: [word for word in text if word not in extra_stopwords_film]
)

df

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,reached,status,description_processed,description_filtered
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,148.172917,1,"[problem, much, entertainment, today, push, un...","[problem, much, entertainment, today, push, un..."
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,106.208130,1,"[million, american, college, student, studied,...","[million, american, college, student, studied,..."
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,5.775000,0,"[full, set, launching, set, show, love, early,...","[full, set, launching, set, show, early, 2000s..."
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD,Film & Video,34.246667,0,"[sleepy, summer, afternoon, stared, void, floa...","[sleepy, summer, afternoon, stared, void, floa..."
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD,Film & Video,101.280000,0,"[hour, pledge, matching, amazing, news, two, g...","[hour, pledge, matching, amazing, news, two, g..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD,Technology,5.927577,0,"[could, launch, website, app, using, voice, im...",NaN
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD,Technology,9.666667,0,"[vfx, oasis, vfx, oasis, industry, focused, vi...",NaN
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD,Technology,10.987179,0,"[sharpener, real, today, modern, sharpener, co...",NaN
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD,Technology,2.143548,0,"[hobbyist, recently, found, wanting, photograp...",NaN


In [11]:
any("film" in text for text in df.loc[df["category"] == "Film & Video", "description_filtered"]) #use to check if it actually took out stopwords like 'film'

TypeError: argument of type 'NoneType' is not iterable

In [18]:
df['category'].unique()

array(['Film & Video', 'Games', 'Music', 'Publishing', 'Technology'],
      dtype=object)

In [23]:

# Keep this version hidden as a comment in case you want the original category-wide loop later
# for category_name, category_df in df.groupby("category"):
#     ...

selected_categories = ['Technology', 'Games', 'Music', 'Publishing', 'Film & Video']  # Add the category names you want to inspect here

category_threshold = 0.45
category_high_freq_words = {}

for category_name in selected_categories:
    category_df = df[df["category"] == category_name].copy()
    docs = category_df["description_processed"]
    if docs.empty:
        print(f"{category_name}: no documents found")
        continue

    category_counter = Counter()
    for doc in docs:
        category_counter.update(set(doc))

    category_table = pd.DataFrame({
        "word": list(category_counter.keys()),
        "doc_freq": list(category_counter.values())
    })

    category_table["doc_freq_ratio"] = category_table["doc_freq"] / len(docs)
    high_freq_words = category_table[category_table["doc_freq_ratio"] > category_threshold]
    high_freq_words = high_freq_words.sort_values("doc_freq_ratio", ascending=False)

    category_high_freq_words[category_name] = high_freq_words
    print(f"\n{category_name}")
    display(high_freq_words)

category_high_freq_words


Technology


,word,doc_freq,doc_freq_ratio
250,use,999,0.670920
371,design,905,0.607790
337,product,854,0.573539
22,experience,813,0.546004
248,designed,802,0.538617
370,see,796,0.534587
20,feature,779,0.523170
311,even,756,0.507723
71,using,746,0.501007
318,without,745,0.500336



Games


,word,doc_freq,doc_freq_ratio
99,game,1199,0.843179
383,play,1018,0.715893
474,every,875,0.615331
1133,player,864,0.607595
504,campaign,827,0.581575
26,experience,813,0.571730
267,back,807,0.567511
272,design,800,0.562588
386,take,789,0.554852
89,come,772,0.542897



Music


...


Publishing


,word,doc_freq,doc_freq_ratio
147,book,938,0.772652
425,love,636,0.523888
681,many,635,0.523064
33,every,613,0.504942
146,come,587,0.483526
887,page,583,0.480231
1212,reward,583,0.480231
498,see,575,0.473641
110,art,568,0.467875
151,cover,557,0.458814



Film & Video


,word,doc_freq,doc_freq_ratio
45,film,1689,0.839881
204,production,1333,0.662854
265,love,1062,0.528095
164,take,1045,0.519642
129,team,1024,0.509199
378,director,1024,0.509199
394,see,1014,0.504227
700,many,1012,0.503232
134,short,995,0.494779
274,show,983,0.488812


{'Technology':            word  doc_freq  doc_freq_ratio
 250         use       999        0.670920
 371      design       905        0.607790
 337     product       854        0.573539
 22   experience       813        0.546004
 248    designed       802        0.538617
 370         see       796        0.534587
 20      feature       779        0.523170
 311        even       756        0.507723
 71        using       746        0.501007
 318     without       745        0.500336
 72         take       735        0.493620
 158       every       727        0.488247
 282      create       713        0.478845
 310        come       708        0.475487
 368        many       704        0.472801
 75   technology       701        0.470786
 215        team       684        0.459369
 300      system       672        0.451310,
 'Games':             word  doc_freq  doc_freq_ratio
 99          game      1199        0.843179
 383         play      1018        0.715893
 474        every       875

In [24]:
# Domain-specific stopwords removal, run only after reviewing the preview above
category_stopwords = {}

for category_name in selected_categories:
    high_freq_words = category_high_freq_words.get(category_name)

    extra_stopwords = set(high_freq_words["word"].tolist())
    category_stopwords[category_name] = extra_stopwords

    category_mask = df["category"] == category_name
    df.loc[category_mask, "description_filtered"] = df.loc[category_mask, "description_processed"].apply(
        lambda text, stopwords=extra_stopwords: [word for word in text if word not in stopwords]
    )

category_stopwords

{'Technology': {'come',
  'create',
  'design',
  'designed',
  'even',
  'every',
  'experience',
  'feature',
  'many',
  'product',
  'see',
  'system',
  'take',
  'team',
  'technology',
  'use',
  'using',
  'without'},
 'Games': {'art',
  'available',
  'back',
  'backer',
  'best',
  'bring',
  'campaign',
  'character',
  'come',
  'create',
  'design',
  'even',
  'every',
  'experience',
  'friend',
  'full',
  'game',
  'keep',
  'level',
  'love',
  'made',
  'many',
  'may',
  'play',
  'player',
  'reward',
  'see',
  'set',
  'stretch',
  'take',
  'team',
  'together',
  'two',
  'unique',
  'use',
  'youll'},
 'Music': {'album',
  'artist',
  'friend',
  'love',
  'many',
  'music',
  'musician',
  'record',
  'recording',
  'song',
  'studio',
  'thank'},
 'Publishing': {'art',
  'book',
  'come',
  'cover',
  'even',
  'every',
  'love',
  'many',
  'page',
  'reward',
  'see',
  'two'},
 'Film & Video': {'bring',
  'come',
  'crew',
  'day',
  'director',
  'experi

In [25]:
df

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,reached,status,description_processed,description_filtered
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,148.172917,1,"[problem, much, entertainment, today, push, un...","[problem, much, entertainment, today, push, un..."
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,106.208130,1,"[million, american, college, student, studied,...","[million, american, college, student, studied,..."
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,5.775000,0,"[full, set, launching, set, show, love, early,...","[full, set, launching, set, early, 2000s, cart..."
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD,Film & Video,34.246667,0,"[sleepy, summer, afternoon, stared, void, floa...","[sleepy, summer, afternoon, stared, void, floa..."
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD,Film & Video,101.280000,0,"[hour, pledge, matching, amazing, news, two, g...","[hour, pledge, matching, amazing, news, two, g..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD,Technology,5.927577,0,"[could, launch, website, app, using, voice, im...","[could, launch, website, app, voice, imagine, ..."
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD,Technology,9.666667,0,"[vfx, oasis, vfx, oasis, industry, focused, vi...","[vfx, oasis, vfx, oasis, industry, focused, vi..."
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD,Technology,10.987179,0,"[sharpener, real, today, modern, sharpener, co...","[sharpener, real, today, modern, sharpener, es..."
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD,Technology,2.143548,0,"[hobbyist, recently, found, wanting, photograp...","[hobbyist, recently, found, wanting, photograp..."


In [26]:
# only at the end 
df.to_csv('Kickstarter_processed.csv')